**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Hardware-Accelerated Scientific Computing

> ⚠️ **Draft — requires an NVIDIA GPU; code cells not executed here.** Authored on a machine without CUDA. Run on a CUDA machine or Colab GPU runtime; an instructor should verify each cell before teaching. Remove this banner after that pass.

The sequel [Intro to GPU Systems](./Intro_GPU.ipynb) promised — twice. There you made the GPU *work*; here you make it *earn its keep*: warps and occupancy, shared memory, and streams — the three levers behind every 10× kernel optimization, organized around the CGMA ratio that notebook introduced.

## 1. Pre-requisites

[Intro to GPU Systems](./Intro_GPU.ipynb) (Numba kernels, host/device model). Same conda environment.

In [ ]:
import numpy as np
from numba import cuda
import math, time
print(cuda.detect())        # confirm your device before proceeding

---
### 🕐 Session 1 of 3 — *Warps, Divergence & Occupancy* (~40 min)
**Goal:** understand the 32-thread execution unit and how divergence/occupancy shape speed.
**Builds on:** [Intro to GPU Systems](./Intro_GPU.ipynb). &nbsp; **Feeds into:** Session 2 (shared memory).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Warps, Divergence & Occupancy</b></summary>

**Timing (~40 min).** 5 min the draft caveat · 12 min what a warp is · 12 min the divergence experiment · 10 min occupancy.

**The practical warning first: this notebook needs an NVIDIA GPU and ships without outputs.** Run every cell yourself on CUDA hardware or a Colab GPU runtime before teaching. `print(cuda.detect())` in the setup cell exists precisely so a missing or unsupported device fails immediately rather than three cells later.

**Open with the correction that reframes everything.** **The GPU does not execute threads — it executes warps.** Thirty-two threads, one instruction, in lockstep. Every design rule in this session is a consequence of that single hardware fact, and a student who holds "thread" as the unit will mispredict performance constantly.

**Then divergence, and derive the cost rather than stating it.** If an `if` splits a warp, the hardware runs **both** paths serially with the inactive threads masked off. So a 50/50 split costs the sum of the two branches — potentially **2× the work** — and a 32-way switch inside one warp costs 32 passes. **The branch is not free even when most threads skip it.**

**Have the room predict the experiment's outcome before running it.** `diverge_bad` branches on `i % 2`, which splits **every** warp; `diverge_good` branches on `(i // 32) % 2`, so **whole warps** take the same path. Identical arithmetic, identical memory traffic, identical launch configuration — **the only difference is the alignment of the condition to the 32-thread boundary.** Ask for a predicted ratio; the honest expectation is close to 2× on the divergent version, tempered by the fact that both `sin` and `cos` still get computed somewhere.

**Praise the benchmark's construction while it is on screen, because this notebook does it right.** There is an explicit **warm-up launch** (Numba JIT-compiles on first call, so timing it measures the compiler), the loop runs **50 iterations** and averages, and `cuda.synchronize()` is called before stopping the clock. **Kernel launches are asynchronous** — omit that synchronize and you time the launch, not the work. Contrast it directly with the CuPy cells in [Intro to GPU Systems](./Intro_GPU.ipynb), which omit exactly this.

**Then occupancy, framed as what it buys rather than what it is.** An SM holds many resident warps and swaps between them at **zero cost** whenever one stalls on memory. **That is how a GPU tolerates 400-cycle memory latency**: it always has something else to run. Occupancy is simply how many warps fit, and it is capped by registers and shared memory per block.

**Make the budget concrete, since "occupancy" is otherwise abstract.** An SM has a fixed register file and a fixed shared-memory allotment. A kernel using many registers per thread, or a block reserving a lot of shared memory, means **fewer blocks resident** and therefore fewer warps to hide latency with. **That is the resource trade behind every launch configuration**, and it is why Session 2's tiling has a real cost as well as a benefit.

**Close with the honest caveat about occupancy, because the naive rule is wrong.** **Higher occupancy is not always better.** A kernel with high arithmetic intensity and good instruction-level parallelism can hit peak at 25% occupancy, while a memory-bound one may need 75%. **Occupancy is a means to latency hiding, not a goal** — and Nsight Compute will tell you which is binding, which is why the conclusion recommends profiling before and after every change.
</details>

## 2. The Warp

💡 **Intuition.** The GPU does not execute threads — it executes **warps**: teams of 32 threads in lockstep, one instruction for all. Two consequences rule kernel design. **Divergence:** an `if` that splits a warp forces both paths to run serially (idle threads masked) — branch on *warp-aligned* conditions when you can. **Occupancy:** each SM juggles many warps and swaps them zero-cost whenever one stalls on memory; enough resident warps = latency *hidden*. Registers and shared memory per block cap how many fit — the resource budget behind every launch config.

In [ ]:
@cuda.jit
def diverge_bad(x, out):
    i = cuda.grid(1)
    if i < x.size:
        if i % 2 == 0:                      # even/odd splits EVERY warp
            out[i] = math.sin(x[i])
        else:
            out[i] = math.cos(x[i])

@cuda.jit
def diverge_good(x, out):
    i = cuda.grid(1)
    if i < x.size:
        if (i // 32) % 2 == 0:              # whole warps take the same path
            out[i] = math.sin(x[i])
        else:
            out[i] = math.cos(x[i])

x_d = cuda.to_device(np.random.rand(2**24).astype(np.float32))
out_d = cuda.device_array_like(x_d)
for name, k in [("warp-splitting", diverge_bad), ("warp-aligned", diverge_good)]:
    k[x_d.size // 256 + 1, 256](x_d, out_d); cuda.synchronize()      # warm up / JIT
    tic = time.perf_counter()
    for _ in range(50): k[x_d.size // 256 + 1, 256](x_d, out_d)
    cuda.synchronize()
    print(f"{name:15s}: {(time.perf_counter()-tic)/50*1e3:.2f} ms")

**What just happened.** Two kernels that compute **exactly the same thing** — `sin` on even indices, `cos` on odd — timed against each other. The only difference is one expression: `i % 2` versus `(i // 32) % 2`.

> ⚠️ This notebook ships **without saved outputs** (see the banner), so the timings are yours to produce on CUDA hardware.

**Predict the direction before running, because the mechanism is derivable.** A warp is **32 threads executing one instruction in lockstep**. `i % 2` alternates within every warp, so **every warp is split**: the hardware runs the `sin` path with odd threads masked off, then the `cos` path with even threads masked off. **Both branches execute, always.** With `(i // 32) % 2`, all 32 threads of a warp share a value, so each warp takes exactly one path and **nothing is wasted**.

**So the expected penalty is close to 2×** — the divergent kernel does the work of both branches for every warp. In practice it is often somewhat less, because `sin` and `cos` share evaluation machinery and the kernel is memory-bound enough that some of the extra arithmetic hides behind loads. **Measure yours; the ratio is a property of your hardware and your branch bodies.**

**The transferable rule is about *alignment*, not about avoiding branches.** Branching is fine; **branching on a condition that splits warps is not**. Whenever the predicate can be made constant across each 32-thread group — by reordering data, by sorting, by padding, or by choosing a different indexing scheme — the divergence disappears at no arithmetic cost. **`i % 2` and `(i // 32) % 2` do the same job and one is free.**

**Note that this generalises well past `if`.** Loops with data-dependent trip counts, early `return`s, and irregular memory access patterns all cause the same lockstep waste. **Any per-thread control flow that differs within a warp costs the union of the paths taken**, and that is the single most common reason a "correct" GPU kernel underperforms.

**Read the benchmark's construction too, because this cell is a model of how to time GPU code.** There is a **warm-up launch** before the clock starts — Numba JIT-compiles on first call, so timing that measures the compiler. The loop runs **50 iterations** and divides, so a single scheduling hiccup does not dominate. And `cuda.synchronize()` is called **before** stopping the clock, because kernel launches are asynchronous and would otherwise return immediately.

**That last point is worth dwelling on, since the same topic contains the counterexample.** The CuPy timings in [Intro to GPU Systems §3.3](./Intro_GPU.ipynb) omit the synchronize and therefore measure launch overhead rather than computation — which is how that notebook arrives at an apparent 75,000× speedup. **Two notebooks, two patterns, one of them right**, and comparing them directly is the most useful benchmarking lesson in this topic.

**One thing the cell does not do, worth flagging as the natural next step.** There is no **correctness check** — no `np.allclose` against a NumPy reference. Both kernels should produce identical output, and verifying that costs one line. **A timing without a correctness check is not a result**, and the Numba example in [Intro to GPU Systems §3.5](./Intro_GPU.ipynb) shows what it looks like when done fully.

---
### 🕐 Session 2 of 3 — *Shared Memory & Tiling* (~40 min)
**Goal:** raise CGMA by staging data in on-chip memory; tile a matrix multiply.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (streams).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Shared Memory & Tiling</b></summary>

**Timing (~40 min).** 10 min the memory hierarchy · 12 min the CGMA arithmetic · 12 min reading the kernel · 6 min the ladder.

**Open with the warehouse-and-workbench image, then immediately put numbers on it.** Global memory is large and slow (hundreds of cycles); **shared memory is a small, fast scratchpad each block controls explicitly** (a few cycles, tens of kilobytes). Unlike a CPU cache, it is **not automatic** — you decide what goes in it and when, which is both the burden and the entire opportunity.

**Derive why naive matmul is hopeless, because the arithmetic is the whole motivation.** A naive kernel computes $C_{ij} = \sum_k A_{ik}B_{kj}$ by reading $2N$ values from global memory to perform $2N$ FLOPs. **CGMA ≈ 0.25 FLOP/byte** — and [Performance Engineering](./Performance_Engineering.ipynb) measured a critical intensity near 34. The kernel is memory-bound by a factor of over 100, and **no amount of arithmetic tuning helps.**

**Then the tiling fix, stated as a reuse count.** Load a $16\times16$ tile of $A$ and of $B$ into shared memory **once**, and every one of the 256 threads reads from it 16 times. **Each global fetch is now amortised over 16 uses, so CGMA rises by roughly $16\times$.** That factor is exactly `TILE`, which is why the tile size is the single most important tuning parameter in the kernel.

**Walk the two `syncthreads()` calls carefully — they are the correctness of the algorithm, not decoration.** The first says *the tile is fully loaded before anyone computes*; the second says *everyone has finished with this tile before it is overwritten*. **Remove either and you get a race condition** that produces plausible, wrong numbers on some runs and correct ones on others. This is also the right moment to correct the description in [Intro to GPU Systems §3.4](./Intro_GPU.ipynb): `syncthreads` synchronises **within a block**, not across the grid.

**Point out the padding branches, because students omit them and then wonder about garbage edges.** The `else: sA[ty, tx] = 0.0` clauses handle matrices whose dimensions are not multiples of `TILE`. **Zeros contribute nothing to the dot product**, so padding is both correct and free — the same idiom as the mask in [Triton](./Triton_Kernels.ipynb), where the compiler handles it for you.

**Connect the tile size to Session 1's occupancy budget, since it closes the loop.** Two $16\times16$ float32 tiles is 2 KB of shared memory per block. Raise `TILE` to 32 and that becomes 8 KB — better reuse, **fewer resident blocks per SM**, less latency hiding. **The optimum is a genuine trade-off, not a maximisation**, and it is why tile size is tuned empirically per architecture.

**Make the closing instruction the actual assignment, because the ladder is the lesson.** Measure three implementations: a naive untiled kernel, this tiled one, and CuPy's cuBLAS call. **Expect roughly 10× from tiling and another several× from cuBLAS** — which uses register blocking, wider tiles, vectorised loads, and tensor cores. Ask the room what the third gap teaches: **hand-written kernels are for operations no library provides**, and reimplementing GEMM is a learning exercise, not a strategy.

**Close by naming what the session actually installed.** Not "use shared memory" but **"count your CGMA before you optimise, and if it is low, buy reuse."** Tiling is one way to buy it; fusion ([Triton](./Triton_Kernels.ipynb) Session 2) is another; changing algorithm is a third. **The diagnosis comes first and the technique follows from it.**
</details>

## 3. The On-Chip Scratchpad

💡 **Intuition.** Global memory is the slow warehouse; **shared memory** is a small, fast workbench each block controls explicitly. The tiling pattern: every thread of a block cooperatively loads a tile of the operands into shared memory (each element fetched from global memory **once**), synchronizes, then all threads reuse the tile many times. Reuse is exactly the CGMA ratio [Intro to GPU Systems §4](./Intro_GPU.ipynb) defined — tiling is how you buy it.

In [ ]:
TILE = 16
@cuda.jit
def matmul_tiled(A, B, C):
    sA = cuda.shared.array((TILE, TILE), dtype=np.float32)
    sB = cuda.shared.array((TILE, TILE), dtype=np.float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    row, col = cuda.grid(2)
    acc = 0.0
    for t in range((A.shape[1] + TILE - 1) // TILE):
        if row < A.shape[0] and t*TILE + tx < A.shape[1]:
            sA[ty, tx] = A[row, t*TILE + tx]
        else: sA[ty, tx] = 0.0
        if col < B.shape[1] and t*TILE + ty < B.shape[0]:
            sB[ty, tx] = B[t*TILE + ty, col]
        else: sB[ty, tx] = 0.0
        cuda.syncthreads()                      # tile fully loaded before anyone computes
        for k in range(TILE):
            acc += sA[ty, k] * sB[k, tx]
        cuda.syncthreads()                      # everyone done before the tile is overwritten
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = acc

N = 1024
A = cuda.to_device(np.random.rand(N, N).astype(np.float32))
B = cuda.to_device(np.random.rand(N, N).astype(np.float32))
C = cuda.device_array((N, N), np.float32)
grid = ((N+TILE-1)//TILE, (N+TILE-1)//TILE)
matmul_tiled[grid, (TILE, TILE)](A, B, C); cuda.synchronize()
tic = time.perf_counter(); matmul_tiled[grid, (TILE, TILE)](A, B, C); cuda.synchronize()
t_tiled = time.perf_counter() - tic
print(f"tiled matmul: {t_tiled*1e3:.1f} ms  ≈ {2*N**3/t_tiled/1e9:.0f} GFLOP/s")
print("compare against a naive (untiled) kernel and against cupy's cuBLAS call — the ladder is the lesson")

**What just happened.** A tiled matrix multiply — $1024^3$, so $2.1$ GFLOP of arithmetic — reported in ms and converted to GFLOP/s, using shared memory to raise reuse.

> ⚠️ This notebook ships **without saved outputs** (see the banner), so the numbers are yours to produce on CUDA hardware.

**Start with the arithmetic that motivates the whole session, because it is checkable on paper.** A **naive** matmul kernel reads $2N$ values from global memory to perform $2N$ FLOPs per output element — **CGMA ≈ 0.25 FLOP/byte**. [Performance Engineering](./Performance_Engineering.ipynb) measured a critical intensity near 34. **The naive kernel is memory-bound by a factor of over 100**, and no amount of arithmetic tuning touches that.

**Tiling changes exactly one thing: the reuse count.** A $16\times16$ tile of $A$ and of $B$ is loaded into shared memory **once**, and each of the 256 threads in the block reads from it **16 times**. Every global fetch is amortised over 16 uses, so **CGMA rises by roughly `TILE` = 16×**. That is why the tile size is the single most important number in the kernel — it *is* the intensity multiplier.

**The two `cuda.syncthreads()` calls are the algorithm's correctness, not decoration.** The first guarantees the tile is **fully loaded before anyone computes**; the second guarantees **everyone has finished with it before it is overwritten** on the next iteration. **Delete either and you get a race**: plausible numbers on some runs, wrong ones on others, with no error message. Note also that this is the correct use of `syncthreads` — **block scope, not grid scope** — which the description in [Intro to GPU Systems §3.4](./Intro_GPU.ipynb) states too broadly.

**The `else: sA[ty, tx] = 0.0` branches are doing real work too.** They pad tiles that hang off the edge of a matrix whose dimensions are not multiples of `TILE`. **Zeros contribute nothing to a dot product**, so padding is correct and free — the same idiom as `mask` in [Triton](./Triton_Kernels.ipynb), where the compiler supplies it for you.

**Note the cost that balances the benefit, because it connects to Session 1.** Two $16\times16$ float32 tiles is **2 KB of shared memory per block**. Raise `TILE` to 32 and it becomes 8 KB — better reuse, but **fewer blocks resident per SM** and therefore fewer warps available to hide memory latency. **Tile size is a genuine trade-off, not a quantity to maximise**, which is why it is tuned empirically per architecture.

**Then do what the last printed line asks, because the ladder is the actual lesson.** Measure three implementations on the same problem: **naive → tiled → cuBLAS** (via CuPy). Expect roughly an order of magnitude from tiling, and **another several× from cuBLAS**, which adds register blocking, wider tiles, vectorised loads, and tensor cores.

**That third gap is the most useful one, and it is worth stating plainly.** After a session of careful work, a vendor library is still substantially faster. **Hand-written kernels are for operations no library provides** — reimplementing GEMM is a learning exercise, not a strategy. The techniques transfer; the specific kernel does not.

**One thing to add before trusting your own numbers.** There is a warm-up launch and a `cuda.synchronize()` before the clock stops — both correct — but **no correctness check**. `np.allclose(C.copy_to_host(), A_host @ B_host, atol=1e-3)` is one line, and given that this kernel's whole risk surface is two barriers and four boundary conditions, **verifying before believing the GFLOP/s is not optional.**

---
### 🕐 Session 3 of 3 — *Streams & Overlap* (~35 min)
**Goal:** overlap transfers with compute; never let either engine idle.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Streams & Overlap</b></summary>

**Timing (~35 min).** 8 min the two engines · 10 min the pipeline picture · 12 min the experiment · 5 min pinned memory.

**Open with the hardware fact that makes streams possible.** A GPU has **separate copy engines and compute units**, and by default your code uses them **one at a time**: upload, then compute, then download, with two of the three idle at every moment. **Streams are independent work queues** that let the scheduler run them concurrently.

**Draw the pipeline, because the picture is the whole idea.** Chunk the data; then while chunk $k$ **computes**, chunk $k{+}1$ **uploads** and chunk $k{-}1$ **downloads**. In steady state all three engines are busy. **This is exactly the overlap-and-pipeline structure from [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb)** — same idea, implemented in silicon rather than in a processing loop.

**State the achievable speedup honestly before running, because the naive expectation is wrong.** Perfect overlap hides **the smaller** of transfer time and compute time — so the ceiling is $(T_{\text{copy}} + T_{\text{compute}}) / \max(T_{\text{copy}}, T_{\text{compute}})$, which is **at most 2×** and only when the two are balanced. **If transfers take 90% of the time, overlap buys about 1.1×.** Have the room compute the bound for the demo's parameters before seeing the result; a 5× answer would mean something is wrong with the measurement.

**Point at `cuda.pinned_array` and explain why it is mandatory rather than an optimisation.** DMA hardware needs physical addresses that will not move. Ordinary host allocations are **pageable** — the [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) may relocate or swap them — so CUDA must first copy into a hidden pinned staging buffer, which is **synchronous and defeats overlap entirely**. **Pinning is what makes an asynchronous copy actually asynchronous.**

**Then the cost of pinning, since it is not free.** Pinned memory cannot be paged out, so it consumes real RAM and pressures the OS. **Pin the buffers you stream through, not your whole dataset** — and note this is exactly what `DataLoader(pin_memory=True)` requests in PyTorch, which is where most students will meet the flag.

**Read the two loops side by side, because the difference is small and total.** Serial: `to_device` → launch → `copy_to_host`, per chunk, each blocking the next. Overlapped: the same three calls with `stream=s` on the copies and `s` in the launch configuration, so **eight independent queues** run concurrently. **The code differs by one argument per line and the execution differs completely.**

**Flag the subtle bug the room should look for, because it is the classic streams mistake.** Any operation *without* a stream argument goes on the **default stream**, which synchronises against everything else and silently serialises your carefully overlapped pipeline. **One missing `stream=s` destroys the entire benefit** with no error and no obvious symptom beyond a disappointing number.

**And note the honest limitation of `heavy`.** It is a synthetic 60-iteration `sin` loop chosen to make compute time comparable to transfer time — because **if compute were negligible, there would be nothing to hide behind**. That is a fair pedagogical choice and worth naming: **the demo constructs the balanced regime where overlap looks best.**

**Close by placing streams among the three levers.** Warps (divergence) is about *instruction* efficiency; shared memory (tiling) is about *memory* efficiency; streams are about *engine* utilisation. **All three are ways of not idling**, and the conclusion's advice is the right one — profile with Nsight before and after each change, because the tool says which lever is binding and the guess usually does not.
</details>

## 4. Streams

💡 **Intuition.** The GPU has separate engines for copying and computing — and by default you use them one at a time. **Streams** are independent work queues: chunk the data, and while chunk $k$ computes, chunk $k{+}1$ uploads and chunk $k{-}1$ downloads. Perfect overlap hides the smaller of transfer/compute time entirely — the [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) pipeline idea, on silicon. Requires *pinned* host memory (`cuda.pinned_array`) so DMA can run without the [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) paging underneath it.

In [ ]:
n_chunks, chunk = 8, 2**22
streams = [cuda.stream() for _ in range(n_chunks)]
host = [cuda.pinned_array(chunk, np.float32) for _ in range(n_chunks)]
for h in host: h[:] = np.random.rand(chunk)

@cuda.jit
def heavy(x):
    i = cuda.grid(1)
    if i < x.size:
        v = x[i]
        for _ in range(60): v = math.sin(v) * 1.0001
        x[i] = v

# serial: one stream does copy→compute→copy for each chunk in turn
tic = time.perf_counter()
for h in host:
    d = cuda.to_device(h)
    heavy[chunk//256+1, 256](d)
    d.copy_to_host(h)
cuda.synchronize(); t_serial = time.perf_counter() - tic

# overlapped: each chunk on its own stream
tic = time.perf_counter()
devs = []
for h, s in zip(host, streams):
    d = cuda.to_device(h, stream=s)
    heavy[chunk//256+1, 256, s](d)
    d.copy_to_host(h, stream=s)
cuda.synchronize(); t_stream = time.perf_counter() - tic
print(f"serial {t_serial*1e3:.0f} ms   streamed {t_stream*1e3:.0f} ms   overlap bought {t_serial/t_stream:.2f}x")

**What just happened.** The same work — eight chunks of 4 million floats, uploaded, transformed, downloaded — done twice: once serially, once with each chunk on its own stream. The printed ratio is what overlap bought.

> ⚠️ This notebook ships **without saved outputs** (see the banner), so the numbers are yours to produce on CUDA hardware.

**Compute the ceiling before reading the result, because the naive expectation is far too optimistic.** Perfect overlap hides **the smaller** of transfer and compute, so the best possible speedup is

$$\frac{T_{\text{copy}} + T_{\text{compute}}}{\max(T_{\text{copy}},\, T_{\text{compute}})} \le 2$$

**At most 2×, and only when the two are balanced.** If transfers dominate at 90% of the time, overlap buys about 1.1×. **A result above 2× would mean the measurement is wrong**, not that the pipeline is exceptional — a useful sanity check to state before the number appears.

**The code difference is one argument per line, and the execution difference is total.** Serial: `to_device` → launch → `copy_to_host`, each blocking the next. Overlapped: the same three calls with `stream=s` on the copies and `s` in the launch configuration. **Eight independent queues, and the scheduler interleaves them** so that while chunk $k$ computes, $k{+}1$ uploads and $k{-}1$ downloads.

**`cuda.pinned_array` is mandatory here, not an optimisation — and the reason is worth knowing.** DMA hardware needs physical addresses that will not move. An ordinary host allocation is **pageable**: the [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) may relocate or swap it at any moment. So CUDA must first copy pageable memory into a hidden pinned staging buffer — **a synchronous step that defeats overlap entirely.** Pinning is what makes an asynchronous copy actually asynchronous.

**And pinning is not free, which is the trade to name.** Pinned pages cannot be swapped out, so they consume real RAM and pressure the OS. **Pin the buffers you stream through, not your whole dataset.** This is exactly what `DataLoader(pin_memory=True)` requests in PyTorch, which is where most students will meet the flag without knowing why it exists.

**Flag the classic mistake, because it produces a disappointing number and no error.** Any operation issued *without* a stream argument lands on the **default stream**, which synchronises against everything else. **One missing `stream=s` silently serialises the whole pipeline.** If your measured speedup is suspiciously close to 1.0, that is the first thing to check.

**Be honest that `heavy` is engineered to make overlap look good.** Sixty iterations of `sin` per element is a synthetic load chosen so that **compute time is comparable to transfer time** — the balanced regime where the ceiling above is largest. With a trivial kernel there would be nothing to hide behind, and with an enormous one the transfers would already be negligible. **The demo constructs its own best case**, which is fine pedagogically and worth stating.

**Finally, place this among the three levers the workshop covered.** Warps and divergence is *instruction* efficiency; shared-memory tiling is *memory* efficiency; streams are *engine* utilisation. **All three are ways of not idling** — and which one binds is a question for Nsight rather than for intuition, which is exactly what the conclusion recommends.

## 5. Conclusion

Think in warps (divergence), feed from the workbench (shared memory raises CGMA), and keep both engines busy (streams). Profile with Nsight before and after each change — the tool tells you which lever is binding.

---
## Where next

- [CUDA in C++](./CUDA_Cpp.ipynb) — the same levers without the Python training wheels.
- [Scaling Neural Networks](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — these ideas at training scale.